# Comprensione del dataset

## Setup

In [ ]:
!apt-get -qq install -y unrar
!pip -q install requests scipy pymupdf matplotlib

## Cosa offre il repository remoto?

In [ ]:
import requests, re

URL = 'https://groups.uni-paderborn.de/kat/BearingDataCenter/'
html = requests.get(URL).text            # scarica la pagina come testo
links = re.findall(r'href="([^"]+)"', html)   # estrae tutti i collegamenti
for link in links:
    print(link)

## Quanti file di ogni tipo?

In [ ]:
import os
from collections import Counter

Counter(os.path.splitext(link)[1] or 'sconosciuto' for link in links)

## Cosa dice il README?

In [ ]:
print(requests.get(URL + 'readme_versions.txt').text)

## Scarichiamo un cuscinetto

In [ ]:
bearing = 'K001'
with open(bearing + '.rar', 'wb') as f:
    f.write(requests.get(URL + bearing + '.rar').content)

## Estraiamo l'archivio

In [ ]:
import subprocess
os.makedirs(bearing, exist_ok=True)
subprocess.run(['unrar', 'x', '-o+', bearing + '.rar', bearing + '/'])

## Cosa c'e' dentro l'archivio?

In [ ]:
for root, dirs, files in os.walk(bearing):
    for f in sorted(files):
        print(os.path.join(root, f))

## Che tipi di file ci sono nell'archivio?

In [ ]:
Counter(os.path.splitext(f)[1] for _, _, files in os.walk(bearing) for f in files)

## Apriamo un file .mat: cosa contiene?

In [ ]:
import glob
from scipy.io import loadmat

mat_files = sorted(glob.glob(os.path.join(bearing, '**', '*.mat'), recursive=True))
file = mat_files[0]
mat = loadmat(file)
key = [k for k in mat if not k.startswith('__')][0]
data = mat[key]
print(os.path.basename(file))
print('variabile:', key)
print('shape:', data.shape)
print('campi:', data.dtype.names)

## Cosa c'e' dentro ogni campo?

In [ ]:
import numpy as np

for field in data.dtype.names:
    value = data[field][0, 0]
    print(field, '|', type(value))
    if isinstance(value, np.ndarray):
        print('  shape:', value.shape, '| dtype:', value.dtype)
        print('  ', value.flatten()[:20] if value.size > 20 else value)
    else:
        print('  ', value)

## Quali segnali (Y) e quali assi (X), con che frequenza?

In [ ]:
mat_s = loadmat(file, simplify_cells=True)[key]
for x in mat_s['X']:
    print('X', x['Raster'], '| campioni:', len(x['Data']))
for y in mat_s['Y']:
    print('Y', y['Name'], '| raster:', y['Raster'], '| campioni:', len(y['Data']))

## I segnali in forma di tabella

In [ ]:
import pandas as pd

pd.DataFrame([
    {'segnale': y['Name'], 'raster': y['Raster'], 'campioni': len(y['Data'])}
    for y in mat_s['Y']
])

## Che aspetto ha un segnale?

In [ ]:
import matplotlib.pyplot as plt

name = 'phase_current_1'   # cambia nome per vedere gli altri segnali
signal = next(y['Data'] for y in mat_s['Y'] if y['Name'] == name)
plt.figure(figsize=(10, 3))
plt.plot(signal[:3000], linewidth=0.6)
plt.title(name)
plt.show()

## Confronto: sano e le tre rotture
Il prefisso del codice indica il tipo: K sano, KA esterno, KI interno, KB entrambe. Prendiamo un cuscinetto per tipo, stesso regime, e guardiamo la corrente.

In [ ]:
examples = {
    'sano': 'K001',
    'pista esterna': 'KA04',
    'pista interna': 'KI04',
    'entrambe': 'KB23',
}
for c in examples.values():
    with open(c + '.rar', 'wb') as f:
        f.write(requests.get(URL + c + '.rar').content)
    os.makedirs(c, exist_ok=True)
    subprocess.run(['unrar', 'x', '-o+', c + '.rar', c + '/'])

In [ ]:
fig, axes = plt.subplots(len(examples), 1, figsize=(10, 9), constrained_layout=True)
for ax, (label, c) in zip(axes, examples.items()):
    f = sorted(glob.glob(os.path.join(c, '**', 'N15_M07_F10_' + c + '_*.mat'), recursive=True))[0]
    m = loadmat(f, simplify_cells=True)
    k = [x for x in m if not x.startswith('__')][0]
    current = next(y['Data'] for y in m[k]['Y'] if y['Name'] == 'phase_current_1')
    ax.plot(current[:3000], linewidth=0.6)
    ax.set_title(label + ' (' + c + ')')
plt.show()

## Cosa dicono i PDF descrittivi?

In [ ]:
import fitz
for pdf in sorted(glob.glob(os.path.join(bearing, '**', '*.pdf'), recursive=True)):
    print(os.path.basename(pdf))
    print(fitz.open(pdf)[0].get_text())